In [7]:
import pandas as pd
import sqlite3
import os

# Load the cleaned dataset
df = pd.read_csv(
    r"F:\Project\Spotify\data\processed\spotify_cleaned.csv"
)

# Create the database folder
os.makedirs(
    r"F:\Project\Spotify\database",
    exist_ok=True
)

# Create and connect to SQLite database
conn = sqlite3.connect(
    r"F:\Project\Spotify\database\spotify.db"
)

# Import the DataFrame into SQLite
df.to_sql(
    'spotify_tracks',
    conn,
    if_exists='replace',
    index=False
)

print("Data successfully imported into SQLite!")



Data successfully imported into SQLite!


## 1. How Many Tracks Are in the Dataset?

Before starting the detailed SQL analysis, I wanted to check the total number of tracks available in my cleaned dataset.

This gives me a quick idea of the size of the dataset I am working with.

In [8]:
query1 = """SELECT COUNT (*) AS Total_Tracks from spotify_tracks;"""
result = pd.read_sql(query1,conn)
result

,Total_Tracks
0,898694


## 2. Which Are the 10 Most Popular Tracks?

I wanted to find the tracks with the highest popularity scores in the dataset.

This helps me see which tracks are currently the most popular and also gives me a look at their artists and genres.

In [11]:
query2 = """SELECT name,track_artists,genres,popularity from spotify_tracks
ORDER BY popularity DESC LIMIT 10;"""
result = pd.read_sql(query2,conn)
result

,name,track_artists,genres,popularity
0,End of Beginning,Djo,"['pov: indie', 'psychedelic pop']",98.0
1,Gata Only,unknown,['reggaeton chileno'],98.0
2,Beautiful Things,unknown,['singer-songwriter pop'],97.0
3,greedy,Tate McRae,['pop'],96.0
4,Cruel Summer,unknown,['pop'],95.0
5,we can't be friends (wait for your love),unknown,['pop'],95.0
6,My Love Mine All Mine,unknown,"['brooklyn indie', 'pov: indie']",94.0
7,Stick Season,unknown,['pov: indie'],94.0
8,CARNIVAL,unknown,[],94.0
9,FE!N (feat. Playboi Carti),unknown,"['hip hop', 'rap', 'slap house']",93.0


## 3. How Are Tracks Distributed Across Different Popularity Levels?

Instead of looking at individual popularity scores, I wanted to divide the tracks into different popularity categories.

This makes it easier to understand how many tracks fall into low, medium, high, and very high popularity levels.

In [14]:
query3 = """SELECT
    case
        when popularity < 25 then 'Low'
        when popularity < 50 then 'Medium'
        when popularity < 75 then 'High'
        else 'Very High'
END AS popularity_category,
COUNT(*) as total_tracks,
ROUND (AVG(popularity),2) as average_popularity
from spotify_tracks
GROUP BY popularity_category
ORDER BY average_popularity;
"""
result = pd.read_sql(query3,conn)
result

,popularity_category,total_tracks,average_popularity
0,Low,507401,6.61
1,Medium,296048,36.29
2,High,91625,57.66
3,Very High,3620,78.86


## 4. Does Musical Mode Affect Track Popularity?

I wanted to compare Major and Minor tracks to see whether there is any difference in their average popularity.

This helps me understand whether musical mode has a noticeable relationship with popularity.

In [16]:
query4 = """SELECT CASE when mode = 1 THEN 'Major'
                        when mode = 0 THEN 'Minor'
                        END AS musical_mode,
                        COUNT (*) AS Total_Tracks,
                        ROUND(AVG(popularity),2) AS average_popularity
                        FROM spotify_tracks
                        GROUP BY mode
                        ORDER BY average_popularity DESC;

"""
result = pd.read_sql(query4,conn)
result

,musical_mode,Total_Tracks,average_popularity
0,Minor,334308,22.28
1,Major,564386,21.65


## 5. What Are the Average Audio Features of Highly Popular Tracks?

From my earlier EDA, I found that some audio features have a relationship with popularity.

Here, I wanted to focus only on highly popular tracks and check their average energy, danceability, valence, acousticness, speechiness, liveness, and instrumentalness.

For this analysis, I am considering tracks with a popularity score of 50 or higher.

In [17]:
query5 = """SELECT COUNT(*) AS Total_Tracks,
    ROUND(AVG(energy),3) AS average_energy,
    ROUND(AVG(danceability),3) AS average_danceability,
    ROUND(AVG(valence),3) AS average_valence,
    ROUND(AVG(acousticness),3) AS average_acousticness,
    ROUND(AVG(speechiness), 3) AS average_speechiness,
    ROUND(AVG(liveness), 3) AS average_liveness,
    ROUND(AVG(instrumentalness), 3) AS average_instrumentalness
FROM spotify_tracks
WHERE popularity >= 50;"""
result = pd.read_sql(query5,conn)
result

,Total_Tracks,average_energy,average_danceability,average_valence,average_acousticness,average_speechiness,average_liveness,average_instrumentalness
0,95245,0.613,0.606,0.494,0.309,0.091,0.186,0.115


## 6. Which Genres Have the Highest Average Popularity?

I wanted to find out which genres have the highest average popularity in the dataset.

Some genres have only a small number of tracks, so I decided to consider only genres with at least 100 tracks. This makes the comparison more meaningful.

In [18]:
query6 = """SELECT
    genres,
    COUNT(*) AS total_tracks,
    ROUND(AVG(popularity), 2) AS average_popularity,
    ROUND(
        AVG(popularity) - (
            SELECT AVG(popularity)
            FROM spotify_tracks
        ),
        2
    ) AS difference_from_overall_average
FROM spotify_tracks
WHERE genres != 'unknown'
GROUP BY genres
HAVING COUNT(*) >= 100
ORDER BY average_popularity DESC
LIMIT 15;"""
result = pd.read_sql(query6,conn)
result

,genres,total_tracks,average_popularity,difference_from_overall_average
0,"['healing hz', 'meditation']",133,61.32,39.44
1,"['hip hop', 'rap', 'slap house']",109,61.06,39.18
2,"['corrido', 'corridos tumbados', 'musica mexic...",166,57.50,35.62
3,"['colombian pop', 'pop reggaeton', 'reggaeton'...",102,56.25,34.37
4,"['corrido', 'corridos tumbados', 'sad sierreno...",232,55.02,33.14
5,['classic oklahoma country'],169,54.52,32.64
6,"['trap latino', 'urbano latino']",138,54.28,32.40
7,"['arrocha', 'sertanejo', 'sertanejo universita...",144,53.85,31.96
8,"['alternative metal', 'nu metal', 'rap metal',...",132,53.58,31.69
9,"['hip hop', 'rap']",210,53.44,31.56


## 7. Which Genres Have the Most Tracks?

Average popularity tells me how well a genre performs, but I also wanted to know which genres appear most often in the dataset.

So, I counted the number of tracks in each genre and selected the top 15.

In [20]:
query7 = """SELECT genres,
    COUNT(*) AS total_tracks from spotify_tracks
    where genres != 'unknown' 
    GROUP BY genres 
    ORDER BY  total_tracks DESC LIMIT 15;
"""
result = pd.read_sql(query7,conn)
result


,genres,total_tracks
0,[],170864
1,"['baroque', 'classical', 'early music', 'germa...",4174
2,['calming instrumental'],3506
3,"['classical', 'classical era']",3481
4,['pop'],3445
5,"['orchestral soundtrack', 'soundtrack']",2658
6,['piano cover'],2607
7,"['dance pop', 'pop']",2226
8,['video game music'],2056
9,['epicore'],1951


## 8. Does Tempo Have Any Relationship With Popularity?

I wanted to check whether tracks with different tempo ranges have different average popularity levels.

To make the comparison easier, I divided the tracks into different BPM ranges and calculated the average popularity for each range.

In [22]:
query8 ="""SELECT
    CASE
        WHEN tempo < 80 THEN 'Below 80 BPM'
        WHEN tempo < 100 THEN '80-99 BPM'
        WHEN tempo < 120 THEN '100-119 BPM'
        WHEN tempo < 140 THEN '120-139 BPM'
        WHEN tempo < 160 THEN '140-159 BPM'
        ELSE '160+ BPM'
    END AS tempo_range,
    COUNT(*) AS total_tracks,
    ROUND(AVG(popularity), 2) AS average_popularity
FROM spotify_tracks
GROUP BY tempo_range
ORDER BY total_tracks DESC;"""
result = pd.read_sql(query8,conn)
result

,tempo_range,total_tracks,average_popularity
0,120-139 BPM,238101,21.96
1,100-119 BPM,184267,21.90
2,80-99 BPM,177818,21.88
3,140-159 BPM,108643,23.14
4,160+ BPM,99362,21.84
5,Below 80 BPM,90503,20.18


## 9. What Are the Top 3 Most Popular Tracks in Each Genre?

I wanted to find the most popular tracks within each genre instead of only finding the overall top tracks.

For this, I am using the `ROW_NUMBER()` window function to rank the tracks separately within each genre.

This also gives me some practice with window functions in SQL.

In [23]:
query9 = """WITH ranked_tracks AS (
    SELECT
        genres,
        name,
        track_artists,
        popularity,
        ROW_NUMBER() OVER (
            PARTITION BY genres
            ORDER BY popularity DESC
        ) AS track_rank
    FROM spotify_tracks
    WHERE genres != 'unknown'
)
SELECT
    genres,
    name,
    track_artists,
    popularity,
    track_rank
FROM ranked_tracks
WHERE track_rank <= 3
ORDER BY genres, track_rank;"""
result = pd.read_sql(query9,conn)
result

,genres,name,track_artists,popularity,track_rank
0,"[""australian children's music"", ""children's fo...",Running On The Spot,Peter Combe,39.0,1
1,"[""australian children's music"", ""children's fo...",I Like to Sing,unknown,26.0,2
2,"[""australian children's music"", ""children's fo...",Spaghetti Bolognaise,unknown,0.0,3
3,"[""australian children's music"", ""children's mu...",Bounce Like a Bunny,unknown,53.0,1
4,"[""australian children's music"", ""children's mu...",Roar Like a Dinosaur,unknown,51.0,2
...,...,...,...,...,...
72624,['zydeco'],On A Night Like This,unknown,31.0,2
72625,['zydeco'],Jambalaya,unknown,31.0,3
72626,[],CARNIVAL,unknown,94.0,1
72627,[],Lose Control,unknown,93.0,2


## 10. How Do Highly Popular Tracks Compare With the Overall Dataset?

Earlier in my Python analysis, I compared popular tracks with the overall dataset.

I wanted to perform a similar comparison using SQL and check whether highly popular tracks have different average audio features.

Here, I am comparing energy, danceability, valence, acousticness, and instrumentalness.

In [24]:
query10 = """SELECT
    'Overall Dataset' AS track_group,
    ROUND(AVG(energy), 3) AS average_energy,
    ROUND(AVG(danceability), 3) AS average_danceability,
    ROUND(AVG(valence), 3) AS average_valence,
    ROUND(AVG(acousticness), 3) AS average_acousticness,
    ROUND(AVG(instrumentalness), 3) AS average_instrumentalness
FROM spotify_tracks
UNION ALL
SELECT
    'Highly Popular Tracks' AS track_group,
    ROUND(AVG(energy), 3),
    ROUND(AVG(danceability), 3),
    ROUND(AVG(valence), 3),
    ROUND(AVG(acousticness), 3),
    ROUND(AVG(instrumentalness), 3)
FROM spotify_tracks
WHERE popularity >= 50;
"""
result = pd.read_sql(query10,conn)
result

,track_group,average_energy,average_danceability,average_valence,average_acousticness,average_instrumentalness
0,Overall Dataset,0.536,0.551,0.437,0.414,0.300
1,Highly Popular Tracks,0.613,0.606,0.494,0.309,0.115


## 11. Which Genres Perform Above the Overall Average?

I wanted to find the genres whose average popularity is higher than the average popularity of the complete dataset.

I am keeping the minimum requirement of 100 tracks per genre so that genres with very few tracks do not affect the result too much.

For this query, I am using a CTE to calculate the overall average first and then compare each genre against it.

In [26]:
query11 = """WITH overall_average AS (
    SELECT AVG(popularity) AS average_popularity
    FROM spotify_tracks
),
genre_analysis AS (
    SELECT
        genres,
        COUNT(*) AS total_tracks,
        AVG(popularity) AS average_popularity
    FROM spotify_tracks
    WHERE genres != 'unknown'
    GROUP BY genres
    HAVING COUNT(*) >= 100
)
SELECT
    ga.genres,
    ga.total_tracks,
    ROUND(ga.average_popularity, 2) AS average_popularity
FROM genre_analysis ga
CROSS JOIN overall_average oa
WHERE ga.average_popularity > oa.average_popularity
ORDER BY ga.average_popularity DESC;"""
result = pd.read_sql(query11,conn)
result

,genres,total_tracks,average_popularity
0,"['healing hz', 'meditation']",133,61.32
1,"['hip hop', 'rap', 'slap house']",109,61.06
2,"['corrido', 'corridos tumbados', 'musica mexic...",166,57.50
3,"['colombian pop', 'pop reggaeton', 'reggaeton'...",102,56.25
4,"['corrido', 'corridos tumbados', 'sad sierreno...",232,55.02
...,...,...,...
811,"['jazz trumpet', 'smooth jazz']",143,21.93
812,"['alternative rock', 'blues rock', 'detroit ro...",168,21.93
813,"['classic rock', 'mellow gold', 'rock', 'soft ...",282,21.90
814,"['anime score', 'japanese soundtrack']",956,21.90


## 12. Which Highly Popular Tracks Have Low Energy?

I wanted to explore whether a track can be highly popular even when it has a relatively low energy level.

For this analysis, I selected tracks with a popularity score of at least 50 and an energy value below 0.40.

This is interesting because it helps show that popularity is not always connected to high-energy music.

In [27]:
query12 = """SELECT
    name,
    track_artists,
    genres,
    popularity,
    energy,
    danceability,
    acousticness,
    instrumentalness
FROM spotify_tracks
WHERE popularity >= 50
  AND energy < 0.40
ORDER BY popularity DESC
LIMIT 20;"""
result = pd.read_sql(query12,conn)
result

,name,track_artists,genres,popularity,energy,danceability,acousticness,instrumentalness
0,My Love Mine All Mine,unknown,"['brooklyn indie', 'pov: indie']",94.0,0.3080,0.504,0.868,0.135000
1,What Was I Made For? [From The Motion Picture ...,unknown,"['art pop', 'pop']",91.0,0.0911,0.444,0.959,0.000001
2,The Night We Met,unknown,"['indie folk', 'stomp and holler']",90.0,0.3690,0.544,0.969,0.279000
3,Evergreen,unknown,['modern folk rock'],89.0,0.2160,0.555,0.557,0.004160
4,Something in the Orange,unknown,['classic oklahoma country'],88.0,0.1920,0.369,0.555,0.000008
5,lovely (with Khalid),unknown,"['art pop', 'pop']",88.0,0.2960,0.351,0.934,0.000000
6,When I Was Your Man,unknown,"['dance pop', 'pop']",87.0,0.2800,0.612,0.932,0.000000
7,All of Me,unknown,"['neo soul', 'pop', 'pop soul', 'urban contemp...",86.0,0.2640,0.422,0.922,0.000000
8,Sparks,Coldplay,"['permanent wave', 'pop']",86.0,0.2680,0.371,0.748,0.051700
9,traitor,unknown,['pop'],86.0,0.3390,0.380,0.691,0.000000
